In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# -----------------------------------------
# configuracion
# -----------------------------------------
CATALOG = "smart_claims_sesion_5"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark = SparkSession.builder.appName("smart_claims").getOrCreate()

# -----------------------------------------
# 0. asegurar gold
# -----------------------------------------
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

# -----------------------------------------
# 1. leer silver
# -----------------------------------------
claims_slv = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.claims_clean")
policies_slv = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.policies_clean")
customers_slv = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean")
telematics_slv = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.telematics_clean")

# -----------------------------------------
# 2. paso A - agregar telematica por vehiculo
# -----------------------------------------
aggregated_telematics = (
    telematics_slv
    .groupBy("chassis_no")
    .agg(
        F.round(F.avg("speed"), 2).alias("avg_speed"),
        F.round(F.avg("latitude"), 6).alias("avg_latitude"),
        F.round(F.avg("longitude"), 6).alias("avg_longitude"),
        F.count("event_timestamp").alias("total_events")
    )
)

aggregated_telematics.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.aggregated_telematics")
print("aggregated_telematics listo")

# -----------------------------------------
# 3. paso B - unir reclamos, polizas y clientes
# -----------------------------------------
customer_claim_policy = (
    claims_slv.alias("c")
    .join(policies_slv.alias("p"),
        F.col("c.policy_no") == F.col("p.POLICY_NO"), "left")
    .join(customers_slv.alias("cu"),
        F.col("p.CUST_ID") == F.col("cu.customer_id"), "left")
    .select(
        F.col("c.claim_no"),
        F.col("c.policy_no"),
        F.col("c.claim_date"),
        F.col("c.severity"),
        F.col("c.total"),
        F.col("c.collision_type"),
        F.col("c.suspicious_activity"),
        F.col("p.CHASSIS_NO"),
        F.col("p.MAKE"),
        F.col("p.MODEL"),
        F.col("p.MODEL_YEAR"),
        F.col("p.PREMIUM"),
        F.col("p.SUM_INSURED"),
        F.col("cu.customer_id"),
        F.col("cu.firstname"),
        F.col("cu.lastname"),
        F.col("cu.address"),
        F.col("cu.date_of_birth")
    )
)

customer_claim_policy.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy")
print("customer_claim_policy listo")

# -----------------------------------------
# 4. paso C - enriquecer con telematica
# -----------------------------------------
customer_claim_policy_telematics = (
    customer_claim_policy.alias("ccp")
    .join(aggregated_telematics.alias("t"),
        F.col("ccp.CHASSIS_NO") == F.col("t.chassis_no"), "left")
    .drop(F.col("t.chassis_no"))
)

customer_claim_policy_telematics.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics")
print("customer_claim_policy_telematics listo")

print("Gold lista")
print(f"tablas creadas: {CATALOG}.{GOLD_SCHEMA}.aggregated_telematics, customer_claim_policy, customer_claim_policy_telematics")

In [0]:
from pyspark.sql import functions as F

CATALOG = "smart_claims_sesion_5"
GOLD_SCHEMA = "gold"

# conteo de registros
print("conteo de registros")
tablas = ["aggregated_telematics", "customer_claim_policy", "customer_claim_policy_telematics"]
for tabla in tablas:
    count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{tabla}").count()
    print(f"{tabla}: {count:,}")

# reclamos sin poliza
sin_poliza = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy") \
    .filter(F.col("POLICY_NO").isNull()).count()
print(f"\nreclamos sin poliza: {sin_poliza}")

# reclamos sin cliente
sin_cliente = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy") \
    .filter(F.col("customer_id").isNull()).count()
print(f"reclamos sin cliente: {sin_cliente}")

# reclamos sin telematica
sin_telematica = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics") \
    .filter(F.col("avg_speed").isNull()).count()
print(f"reclamos sin telematica: {sin_telematica}")

# muestra final
print("\nmuestra tabla final")
spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics") \
    .select("claim_no", "firstname", "lastname", "POLICY_NO", "CHASSIS_NO", "severity", "total", "avg_speed") \
    .limit(5).display()